# Sentinel-1 SLC → gamma0 RTC VV/VH on a common grid, via ASF HyP3

Takes four SLC acquisitions and an AOI, has ASF HyP3 produce radiometrically
terrain-corrected gamma0, then regrids everything onto one shared grid and writes
a GeoTIFF per date with two named bands (`gamma0_VV`, `gamma0_VH`).

**Why HyP3 rather than a local SAR stack.** OTB has no terrain flattening and is
not packaged for Windows; ISCE2/ISCE3 are Linux-only and do not calibrate
radiometrically. HyP3 runs the processing on ASF's servers and returns exactly the
product asked for: gamma0 RTC, VV and VH, GeoTIFF, UTM.

## How to use it

**1. Install the dependencies.**

```
conda install -c conda-forge asf_search hyp3_sdk rasterio geopandas numpy
```

**2. Get a NASA Earthdata Login** (free, <https://urs.earthdata.nasa.gov>) and
store the credentials in a netrc file in your home directory — on Windows,
`C:/Users/<you>/.netrc`:

```
machine urs.earthdata.nasa.gov
login <username>
password <password>
```

`requests` finds that file on its own, by matching the host name. Nothing else to
configure.

**3. Fill in the parameters cell:** `MODE`, the four acquisitions (local paths or
bare granule names), the AOI, and the two output directories (default values implemented). Point those outside
the repository if you can — the products are heavy.

**4. Run cells 1 to 5 and read them.** Cell 4 must report `OK` for all four
acquisitions; cell 5 lists the granules that will actually be submitted. Nothing
has been spent at this stage — both cells only query the free catalogue.

**5. Run cell 6** and compare the credit balance with the number of granules from
cell 5. This is the moment to drop to 20 or 30 m if the budget is tight.

**6. Run cell 7 once.** This is the submission, and the only cell that costs
credits. If you later restart the kernel, do **not** run it again: recover the
jobs with `batch = hyp3.find_jobs(name=JOB_NAME)`.

**7. Run cell 8.** It blocks until ASF is done — minutes to a few hours — then
downloads and extracts. Safe to re-run: it costs bandwidth, not credits. Do not
postpone it too long, HyP3 deletes the products after about two weeks.

**8. Run cells 9 to 11.** Purely local. Change `PIXEL_SIZE` or the AOI and replay
them as often as you like, with no reprocessing and no cost.

Results land in `OUT_DIR`: one GeoTIFF per acquisition, bands `gamma0_VV` and
`gamma0_VH`, all sharing the same grid so they stack pixel to pixel.

## Two modes

Set by `MODE` in the parameters cell.

| | `"scene"` | `"burst"` |
| --- | --- | --- |
| what is processed | the whole IW scene | only the bursts touching the AOI |
| jobs | 1 per date | 1 per burst *and* per polarisation |
| cost and wall time | high | much lower per job |
| granule ids | the local product names | found by `asf_search` from the AOI |

Burst granules carry ESA's *absolute* burst ids, unrelated to the product-local
numbering of `polygon_to_swaths_bursts`. Rather than converting between the two,
burst mode asks `asf_search` which bursts intersect the AOI — and cell 4 keeps our
own selection as a cross-check.

## What each cell does

The **ASF** column separates free catalogue queries from the calls that consume
credits, so nothing is spent before everything checkable has been checked.

| # | What it does | ASF |
| --- | --- | --- |
| 1 | This overview. | — |
| 2 | Imports, and puts the sibling `polygon_to_swaths_bursts` folder on `sys.path`. | — |
| 3 | Every parameter: mode, the four acquisitions, the AOI, directories, pixel size, job name, RTC options. | — |
| 4 | Coverage check: does each acquisition really reach the AOI, and through which swaths and bursts. Falls back to the scene footprint when the product is not available locally. | catalogue, free |
| 5 | Builds the list of granules to submit — product names in scene mode, one `asf_search` burst query per acquisition in burst mode. | catalogue, free |
| 6 | Connects to HyP3 through the netrc, prints the credit balance and the real `submit_rtc_job` signature. | yes |
| 7 | Submits one RTC job per granule under `JOB_NAME`. **This is the cell that spends credits.** | yes |
| 8 | Waits for the jobs to finish, downloads the products and extracts the archives. | yes |
| 9 | Computes the common output grid: AOI bounding box in UTM, snapped to the pixel size. | — |
| 10 | Regrids every downloaded raster onto that grid, merges the tiles of a same date, writes one 2-band GeoTIFF per acquisition. | — |
| 11 | Reports the outputs: grid of each file and share of valid pixels. | — |

**To verify on first run:** `submit_rtc_job` keywords differ between scene and
burst jobs, and between `hyp3_sdk` versions. Cell 6 prints the real signature.

In [ ]:
import inspect
import sys
import zipfile
from datetime import datetime, timedelta
from pathlib import Path

import asf_search as asf
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import Resampling, reproject

# polygon_to_swaths_bursts is a sibling folder: make it importable
TOOLS = Path.cwd().parent / "polygon_to_swaths_bursts"
sys.path.insert(0, str(TOOLS))
from polygon_to_swaths_bursts import get_intersecting_bursts, parse_polygon

In [ ]:
# --- Parameters: adapt to your data ---

MODE = "burst"   # "burst" (only what the AOI touches) or "scene" (whole images)

# The four SLC acquisitions to process. ASF works on its own copy — nothing is
# uploaded — so each entry may be either a local .SAFE / .zip path or just the
# bare granule name:
#     "S1B_IW_SLC__1SDV_20170804T215105_20170804T215131_006796_00BF5A_B333"
# Only the AOI coverage check in cell 4 opens the local files, and it falls back
# to the ASF catalogue for the entries it cannot find. Scene mode submits these
# names as granules; burst mode uses their acquisition times to find the bursts.
SLC_PATHS = [
    "C:/Users/guigu/Documents/pro_asus/vigisar/data/data_raw/zta10/S1B_IW_SLC__1SDV_20170804T215105_20170804T215131_006796_00BF5A_B333.SAFE",
    "C:/path/to/second.SAFE",
    "C:/path/to/third.SAFE",
    "C:/path/to/fourth.SAFE",
]

# Area of interest, lon/lat (EPSG:4326): inline WKT, or a WKT / GeoJSON file path
AOI = "POLYGON ((-54.383019 5.252325, -54.546861 5.252325, -54.546861 5.128303, -54.383019 5.128303, -54.383019 5.252325))"

# Where the files land. A relative path resolves against this notebook's folder;
# an absolute one ("C:/data/hyp3") works just as well and is the better habit,
# since these products are heavy and do not belong in the repo. Both directories
# are created on the fly, missing parent folders included.
DOWNLOAD_DIR = "hyp3_downloads"   # raw HyP3 products: zips, then extracted
OUT_DIR = "output"                # the final 2-band GeoTIFFs

PIXEL_SIZE = 10.0                 # metres; HyP3 RTC accepts 10, 20 or 30
POLARISATIONS = ["VV", "VH"]

# Free-text label stamped on every job, and the only handle to find them again
# later with hyp3.find_jobs(name=...). It is not unique server-side, so include
# what distinguishes one submission from another: two runs sharing a name come
# back mixed together.
JOB_NAME = f"zta10-{MODE}-{int(PIXEL_SIZE)}m"

# gamma0 + power is the standard pair for analysis: keep the linear scale here
# and convert to dB only for display. Burst jobs accept fewer options than scene
# jobs — check against the signature printed in cell 6.
RTC_OPTIONS = dict(
    radiometry="gamma0",
    scale="power",
    resolution=int(PIXEL_SIZE),
)
if MODE == "scene":
    RTC_OPTIONS |= dict(
        dem_name="copernicus",
        speckle_filter=False,   # filtering is a downstream choice, keep raw data
        dem_matching=False,     # can degrade geolocation over flat or wet terrain
    )


def acquisition_window(slc_path):
    """(start, stop) datetimes read from a Sentinel-1 product name."""
    parts = Path(slc_path).stem.split("_")
    return (
        datetime.strptime(parts[5], "%Y%m%dT%H%M%S"),
        datetime.strptime(parts[6], "%Y%m%dT%H%M%S"),
    )


DATES = [acquisition_window(p)[0].strftime("%Y%m%d") for p in SLC_PATHS]

In [ ]:
# --- Does the AOI really fall inside each acquisition? ---
# Scene mode: this is the only guard against paying to process a product that
# misses the AOI. Burst mode: asf_search selects by intersection anyway, so it is
# just a cross-check. With a local product the check is burst by burst; without
# one it falls back to the scene footprint from the ASF catalogue — a free query,
# no download, which also confirms that the granule name exists.
aoi_geom = parse_polygon(AOI)

for slc in SLC_PATHS:
    name = Path(slc).stem

    if Path(slc).exists():
        _, summary = get_intersecting_bursts(slc, AOI, coarse=True)
        if summary:
            detail = ", ".join(f"{sw} {bursts}" for sw, bursts in sorted(summary.items()))
            print(f"OK      {name[:58]}\n        {detail}")
        else:
            print(f"NO DATA {name[:58]} — the AOI is outside this product")
        continue

    results = asf.granule_search([name])
    if not results:
        print(f"UNKNOWN {name[:58]} — no such granule in the ASF catalogue")
    elif any(parse_polygon(r.geometry).intersects(aoi_geom) for r in results):
        print(f"OK      {name[:58]}\n        AOI inside the scene footprint (catalogue)")
    else:
        print(f"NO DATA {name[:58]} — the AOI is outside this scene")

In [ ]:
# --- The granules to submit ---
aoi_wkt = parse_polygon(AOI).wkt

if MODE == "scene":
    # The ASF scene identifier is the product name without its extension
    GRANULES = [Path(p).stem for p in SLC_PATHS]
else:
    # Ask ASF which burst granules intersect the AOI, one acquisition at a time.
    # A one-minute margin around the product times isolates that acquisition.
    GRANULES = []
    for slc in SLC_PATHS:
        start, stop = acquisition_window(slc)
        results = asf.search(
            platform=asf.PLATFORM.SENTINEL1,
            processingLevel="BURST",
            intersectsWith=aoi_wkt,
            start=start - timedelta(minutes=1),
            end=stop + timedelta(minutes=1),
            polarization=POLARISATIONS,
        )
        found = sorted(r.properties["fileID"] for r in results)
        print(f"{Path(slc).stem[:52]}: {len(found)} bursts")
        for burst in found:
            print("   ", burst)
        GRANULES += found

print(f"\n{len(GRANULES)} granules to submit in {MODE} mode")

In [ ]:
# --- Connect to HyP3 and check what this version accepts ---
import hyp3_sdk as sdk

print("hyp3_sdk", sdk.__version__)

# Credentials are read from the netrc file in your home directory
# (C:/Users/<you>/.netrc, or _netrc — requests accepts either):
#
#     machine urs.earthdata.nasa.gov
#     login <earthdata username>
#     password <earthdata password>
#
# Called without arguments, HyP3 lets requests pick them up from there. Pass
# prompt="password" or prompt="token" instead to be asked interactively.
netrc = next(
    (p for p in (Path.home() / ".netrc", Path.home() / "_netrc") if p.exists()), None
)
if netrc is None:
    raise FileNotFoundError(
        f"no .netrc or _netrc found in {Path.home()} — create one as shown above, "
        'or switch to sdk.HyP3(prompt="password")'
    )
print("credentials from", netrc)

hyp3 = sdk.HyP3()

info = hyp3.my_info()
print("user:", info.get("user_id"), "| remaining credits:", info.get("remaining_credits"))

# Compare RTC_OPTIONS above with the keywords this version really accepts
print("\nsubmit_rtc_job", inspect.signature(hyp3.submit_rtc_job))

In [ ]:
# --- Submit one RTC job per granule ---
batch = sdk.Batch()
for granule in GRANULES:
    batch += hyp3.submit_rtc_job(granule, name=JOB_NAME, **RTC_OPTIONS)

print(f"{len(batch)} jobs submitted under the name {JOB_NAME!r}")

# Already submitted earlier? Recover them instead of paying twice:
#   batch = hyp3.find_jobs(name=JOB_NAME)

In [ ]:
# --- Wait for processing, then download and extract (minutes to a few hours) ---
batch = hyp3.watch(batch)

download_dir = Path(DOWNLOAD_DIR)
download_dir.mkdir(parents=True, exist_ok=True)
zips = batch.download_files(location=download_dir)

for archive in zips:
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(download_dir)
    print("extracted:", Path(archive).name)

In [ ]:
# --- The common output grid: AOI bbox in UTM, snapped to PIXEL_SIZE ---
aoi_gs = gpd.GeoSeries([parse_polygon(AOI)], crs="EPSG:4326")
UTM_CRS = aoi_gs.estimate_utm_crs()
minx, miny, maxx, maxy = aoi_gs.to_crs(UTM_CRS).total_bounds

# Snapping to round coordinates guarantees that every date, and any date added
# later, lands on exactly the same pixel centres.
ULX = np.floor(minx / PIXEL_SIZE) * PIXEL_SIZE
ULY = np.ceil(maxy / PIXEL_SIZE) * PIXEL_SIZE
SIZE_X = int(np.ceil((maxx - ULX) / PIXEL_SIZE))
SIZE_Y = int(np.ceil((ULY - miny) / PIXEL_SIZE))
TRANSFORM = from_origin(ULX, ULY, PIXEL_SIZE, PIXEL_SIZE)

print(f"{UTM_CRS.name} (EPSG:{UTM_CRS.to_epsg()})")
print(f"{SIZE_X} x {SIZE_Y} px at {PIXEL_SIZE} m, upper-left ({ULX}, {ULY})")

In [ ]:
def find_rtc_bands(date, pol, search_dir):
    """HyP3 RTC GeoTIFFs for one date and polarisation.

    One file in scene mode, one per burst in burst mode — HyP3 names its outputs
    after the input granule, so both carry the acquisition date.
    """
    matches = [
        p for p in Path(search_dir).rglob(f"*_{pol}.tif")
        if date in p.name and "rgb" not in p.name.lower()
    ]
    if not matches:
        raise FileNotFoundError(f"no {pol} RTC file for {date} under {search_dir}")
    return sorted(matches)


def on_common_grid(src_path):
    """Resample one RTC GeoTIFF onto the shared grid. Nodata becomes NaN."""
    target = np.full((SIZE_Y, SIZE_X), np.nan, dtype="float32")
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=target,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=TRANSFORM,
            dst_crs=UTM_CRS,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
    return target


def merge_on_grid(paths):
    """Regrid several tiles and merge them: average where they overlap."""
    total = np.zeros((SIZE_Y, SIZE_X), dtype="float64")
    count = np.zeros((SIZE_Y, SIZE_X), dtype="uint16")
    for path in paths:
        values = on_common_grid(path)
        valid = np.isfinite(values) & (values > 0)
        total[valid] += values[valid]
        count[valid] += 1
    merged = np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)
    return merged.astype("float32")


outputs = []
for slc, date in zip(SLC_PATHS, DATES):
    bands = []
    for pol in POLARISATIONS:
        tiles = find_rtc_bands(date, pol, DOWNLOAD_DIR)
        print(f"{date} {pol}: {len(tiles)} tile(s)")
        bands.append(merge_on_grid(tiles))

    out_path = Path(OUT_DIR) / f"{Path(slc).stem}_gamma0.tif"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        out_path, "w", driver="GTiff",
        height=SIZE_Y, width=SIZE_X, count=len(POLARISATIONS),
        dtype="float32", crs=UTM_CRS, transform=TRANSFORM, nodata=np.nan,
        compress="deflate", tiled=True,
    ) as dst:
        for index, (pol, band) in enumerate(zip(POLARISATIONS, bands), start=1):
            dst.write(band, index)
            dst.set_band_description(index, f"gamma0_{pol}")

    print("written:", out_path)
    outputs.append(out_path)

In [ ]:
# --- Check the outputs: same grid everywhere, enough valid pixels ---
for path in outputs:
    with rasterio.open(path) as src:
        print(path.name)
        print(f"    {src.crs}, {src.width}x{src.height} px, "
              f"pixel {src.transform.a:g} m")
        for index in range(1, src.count + 1):
            data = src.read(index)
            valid = np.isfinite(data) & (data > 0)
            print(f"    {src.descriptions[index - 1]}: "
                  f"{valid.sum() / data.size:.0%} valid pixels")